In [1]:
import pandas as pd
import os
import csv
import re

OUTPUT_DIR = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Pasta de saída: {OUTPUT_DIR}')
print('Bibliotecas carregadas com sucesso ✅')

Pasta de saída: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas
Bibliotecas carregadas com sucesso ✅


In [2]:
INPUT_LOC  = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Localizacao.csv'
OUTPUT_LOC = os.path.join(OUTPUT_DIR, 'Localizacao_tratada.xlsx')

with open(INPUT_LOC, 'r', encoding='utf-8-sig') as f:
    amostra = ''.join([f.readline() for _ in range(10)])
try:
    dialeto = csv.Sniffer().sniff(amostra, delimiters=',;\t')
    sep_detectado = dialeto.delimiter
except csv.Error:
    sep_detectado = ','
print(f'Separador detectado: {repr(sep_detectado)}')

df_raw = pd.read_csv(
    INPUT_LOC, sep=sep_detectado, header=None,
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)

header_row_idx = None
for i, row in df_raw.iterrows():
    if any('ID' in str(v) and 'ocaliza' in str(v) for v in row.values):
        header_row_idx = i
        break
if header_row_idx is None:
    header_row_idx = 2
print(f'Cabeçalho detectado na linha: {header_row_idx + 1}')

df = pd.read_csv(
    INPUT_LOC, sep=sep_detectado, header=header_row_idx,
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)
print('Colunas originais:', df.columns.tolist())
print('Shape inicial:', df.shape)
df.head(10)

Separador detectado: ';'
Cabeçalho detectado na linha: 3
Colunas originais: ['ID Localização', 'Tipo Localização', 'Cidade', 'Estado', 'País']
Shape inicial: (679, 5)


,ID Localização,Tipo Localização,Cidade,Estado,País
0,Continente:,Asia,,,
1,269,Country/Region,NULL,NULL,Armenia
2,270,Country/Region,NULL,NULL,Australia
3,271,Country/Region,NULL,NULL,Bhutan
4,272,Country/Region,NULL,NULL,China
5,273,Country/Region,NULL,NULL,India
6,274,Country/Region,NULL,NULL,Iran
7,275,Country/Region,NULL,NULL,Japan
8,276,Country/Region,NULL,NULL,Kyrgyzstan
9,277,Country/Region,NULL,NULL,Pakistan


In [3]:
df = df.loc[:, df.columns.str.strip() != '']
df = df.iloc[:, :5]
df.columns = ['ID Localização', 'Tipo Localização', 'Cidade', 'Estado', 'País']

continente_atual = ''
lista_continente = []
linhas_separador = []

for idx, row in df.iterrows():
    id_val   = str(row['ID Localização']).strip()
    tipo_val = str(row['Tipo Localização']).strip()
    if 'continente' in id_val.lower():
        partes = id_val.split(':', 1)
        continente_atual = partes[1].strip() if len(partes) > 1 and partes[1].strip() else tipo_val
        linhas_separador.append(idx)
        lista_continente.append('')
    else:
        lista_continente.append(continente_atual)

df['Continente'] = lista_continente
df = df.drop(index=linhas_separador).reset_index(drop=True)
df = df[df['ID Localização'].str.strip() != ''].reset_index(drop=True)

print(f'Linhas de separador removidas: {len(linhas_separador)}')
print('Shape após limpeza:', df.shape)
print('Continentes encontrados:', df['Continente'].unique().tolist())
df.head(10)

Linhas de separador removidas: 3
Shape após limpeza: (671, 6)
Continentes encontrados: ['Asia', 'Europe', 'North America']


,ID Localização,Tipo Localização,Cidade,Estado,País,Continente
0,269,Country/Region,NULL,NULL,Armenia,Asia
1,270,Country/Region,NULL,NULL,Australia,Asia
2,271,Country/Region,NULL,NULL,Bhutan,Asia
3,272,Country/Region,NULL,NULL,China,Asia
4,273,Country/Region,NULL,NULL,India,Asia
5,274,Country/Region,NULL,NULL,Iran,Asia
6,275,Country/Region,NULL,NULL,Japan,Asia
7,276,Country/Region,NULL,NULL,Kyrgyzstan,Asia
8,277,Country/Region,NULL,NULL,Pakistan,Asia
9,278,Country/Region,NULL,NULL,Singapore,Asia


In [4]:
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
df.replace('NULL', '', inplace=True)

with pd.ExcelWriter(OUTPUT_LOC, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Localizacao')
    ws = writer.sheets['Localizacao']
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.number_format = '@'

print(f'✅ Arquivo salvo em: {OUTPUT_LOC}')
print(f'Total de registros exportados: {len(df)}')
df.tail(5)

✅ Arquivo salvo em: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Localizacao_tratada.xlsx
Total de registros exportados: 671


,ID Localização,Tipo Localização,Cidade,Estado,País,Continente
666,948,City,Waterbury,Connecticut,United States,North America
667,949,City,Waukesha,Wisconsin,United States,North America
668,950,City,Wheat Ridge,Colorado,United States,North America
669,951,City,Winchester,Virginia,United States,North America
670,952,City,Worcester,Massachusetts,United States,North America


In [5]:
INPUT_META  = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Metas.xlsx'
OUTPUT_META = os.path.join(OUTPUT_DIR, 'Metas_tratada.xlsx')

ABAS_ANOS = {'2017': 2017, '2018': 2018, '2019': 2019}
frames = []

for nome_aba, ano in ABAS_ANOS.items():
    print(f'\n── Processando aba: {nome_aba} ──')

    df_raw = pd.read_excel(INPUT_META, sheet_name=nome_aba, header=None, dtype=str)

    header_idx = None
    for i, row in df_raw.iterrows():
        if any('Asia' in str(v) or 'Europe' in str(v) or 'North America' in str(v) for v in row.values):
            header_idx = i
            break
    if header_idx is None:
        header_idx = 1
    print(f'  Cabeçalho na linha: {header_idx + 1}')

    df_aba = pd.read_excel(INPUT_META, sheet_name=nome_aba, header=header_idx)

    df_aba.columns = ['Categoria'] + list(df_aba.columns[1:])
    df_aba = df_aba.loc[:, ~df_aba.columns.astype(str).str.contains('^Unnamed')]

    df_aba['Categoria'] = df_aba['Categoria'].astype(str).str.strip()
    df_aba = df_aba[df_aba['Categoria'].str.lower() != 'total']
    df_aba = df_aba[~df_aba['Categoria'].isin(['', 'nan'])].reset_index(drop=True)
    print(f'  Registros (sem Total): {len(df_aba)}')

    continentes = [c for c in df_aba.columns if c != 'Categoria']
    df_melt = df_aba.melt(
        id_vars='Categoria',
        value_vars=continentes,
        var_name='Continente',
        value_name='Meta (R$)'
    )

    df_melt['Ano'] = str(ano)
    df_melt['Meta (R$)'] = pd.to_numeric(df_melt['Meta (R$)'], errors='coerce')

    frames.append(df_melt)
    print(f'  Aba {nome_aba} OK | Exemplo Meta: R$ {df_melt["Meta (R$)"].iloc[0]:,.2f}')

df_meta = pd.concat(frames, ignore_index=True)
df_meta = df_meta[['Ano', 'Categoria', 'Continente', 'Meta (R$)']]

print(f'\n✅ Total consolidado: {len(df_meta)} registros')
print('Anos:', df_meta['Ano'].unique().tolist())
print('Categorias:', df_meta['Categoria'].unique().tolist())
print('Continentes:', df_meta['Continente'].unique().tolist())
df_meta.head(15)

for col in ['Ano', 'Categoria', 'Continente']:
    df_meta[col] = df_meta[col].astype(str).str.strip()

df_meta['Meta (R$)'] = pd.to_numeric(df_meta['Meta (R$)'], errors='coerce')

FORMATO_MOEDA = 'R$ #,##0.00'

with pd.ExcelWriter(OUTPUT_META, engine='openpyxl') as writer:
    df_meta.to_excel(writer, index=False, sheet_name='Metas')
    ws = writer.sheets['Metas']
    col_names = [cell.value for cell in ws[1]]
    meta_col_idx = col_names.index('Meta (R$)') + 1
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if cell.column == meta_col_idx:
                cell.number_format = FORMATO_MOEDA
            else:
                cell.number_format = '@'

print(f'✅ Arquivo salvo em: {OUTPUT_META}')
print(f'Total de registros exportados: {len(df_meta)}')
df_meta


── Processando aba: 2017 ──
  Cabeçalho na linha: 2
  Registros (sem Total): 3
  Aba 2017 OK | Exemplo Meta: R$ 364,664.00

── Processando aba: 2018 ──
  Cabeçalho na linha: 2
  Registros (sem Total): 3
  Aba 2018 OK | Exemplo Meta: R$ 784,306.55

── Processando aba: 2019 ──
  Cabeçalho na linha: 2
  Registros (sem Total): 3
  Aba 2019 OK | Exemplo Meta: R$ 1,021,634.40

✅ Total consolidado: 27 registros
Anos: ['2017', '2018', '2019']
Categorias: ['Audio', 'Computers', 'TV and Video']
Continentes: ['Asia', 'Europe', 'North America']
✅ Arquivo salvo em: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Metas_tratada.xlsx
Total de registros exportados: 27


,Ano,Categoria,Continente,Meta (R$)
0,2017,Audio,Asia,364664.00
1,2017,Computers,Asia,13340355.00
2,2017,TV and Video,Asia,1283576.00
3,2017,Audio,Europe,1000400.00
4,2017,Computers,Europe,33190190.00
5,2017,TV and Video,Europe,8433615.00
6,2017,Audio,North America,390579.00
7,2017,Computers,North America,15337020.00
8,2017,TV and Video,North America,1021565.00
9,2018,Audio,Asia,784306.55


In [ ]:
#PRODUTO#

INPUT_PROD  = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Produto.csv'
OUTPUT_PROD = os.path.join(OUTPUT_DIR, 'Produto_tratado.xlsx')

# ─── Detectar separador ───────────────────────────────────────────────────────
with open(INPUT_PROD, 'r', encoding='utf-8-sig') as f:
    amostra = ''.join([f.readline() for _ in range(10)])
try:
    dialeto = csv.Sniffer().sniff(amostra, delimiters=',;\t')
    sep_prod = dialeto.delimiter
except csv.Error:
    sep_prod = ','
print(f'Separador detectado: {repr(sep_prod)}')

# ─── Leitura bruta para detectar linha do cabeçalho ──────────────────────────
df_raw = pd.read_csv(
    INPUT_PROD, sep=sep_prod, header=None,
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)

header_idx = None
for i, row in df_raw.iterrows():
    vals = [str(v).strip().lower() for v in row.values]
    if 'produto' in vals and 'subcategoria' in vals:
        header_idx = i
        break
if header_idx is None:
    header_idx = 2
print(f'Cabeçalho detectado na linha: {header_idx + 1}')

# ─── Re-leitura com cabeçalho correto ─────────────────────────────────────────
df = pd.read_csv(
    INPUT_PROD, sep=sep_prod, header=header_idx,
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)

# Manter apenas as 2 colunas relevantes
df = df.iloc[:, :2]
df.columns = ['Produto_Raw', 'Subcategoria_Raw']

print('Shape inicial:', df.shape)
df.head(10)

import re

# ─── Cores conhecidas para extração (opcional) ────────────────────────────────
CORES = {
    'silver', 'black', 'white', 'blue', 'red', 'green', 'brown',
    'grey', 'gray', 'gold', 'orange', 'pink', 'yellow', 'purple'
}

def extrair_campos_produto(nome_raw):
    """
    Recebe: 'Produto # 116 - Adventure Works 20" CRT TV E15 Silver'
    Retorna dict com: ID_Produto, Nome_Produto, Tipo_Produto, Modelo, Cor, Tamanho
    """
    resultado = {
        'ID_Produto': '', 'Nome_Produto': '',
        'Tipo_Produto': '', 'Modelo': '',
        'Cor': '', 'Tamanho': ''
    }

    nome_raw = str(nome_raw).strip()

    # Extrair ID — número após 'Produto #'
    m_id = re.search(r'Produto\s*#\s*(\d+)', nome_raw, re.IGNORECASE)
    if m_id:
        resultado['ID_Produto'] = m_id.group(1)

    # Separar pelo primeiro ' - ' para isolar o nome descritivo
    partes = nome_raw.split(' - ', 1)
    if len(partes) < 2:
        resultado['Nome_Produto'] = nome_raw
        return resultado

    descricao = partes[1].strip()   # ex: 'Adventure Works 20" CRT TV E15 Silver'
    resultado['Nome_Produto'] = descricao

    tokens = descricao.split()

    # Extrair Cor — última palavra se for cor conhecida (opcional)
    if tokens and tokens[-1].lower() in CORES:
        resultado['Cor'] = tokens[-1]
        tokens = tokens[:-1]

    # Extrair Modelo — último token com padrão letra(s)+número(s) ex: E15, M110, E200
    modelo_idx = None
    for i in range(len(tokens) - 1, -1, -1):
        if re.fullmatch(r'[A-Za-z]+\d+', tokens[i]):
            resultado['Modelo'] = tokens[i]
            modelo_idx = i
            break
    if modelo_idx is not None:
        tokens = tokens[:modelo_idx]

    # Extrair Tamanho — token com padrão número+" ex: 20", 13" (opcional)
    tamanho_idx = None
    for i, t in enumerate(tokens):
        if re.fullmatch(r'\d+"', t):
            resultado['Tamanho'] = t
            tamanho_idx = i
            break
    if tamanho_idx is not None:
        tokens = tokens[:tamanho_idx] + tokens[tamanho_idx+1:]

    # Tipo_Produto — o que sobrar após remover a Marca (primeiras palavras)
    # A marca termina onde começa o tipo — heurística: tipo é o que vem
    # depois das palavras da marca (palavras capitalizadas sem número)
    # Identificar onde a marca termina: pegar tokens sem dígitos até encontrar
    # um token que seja parte do nome do tipo (palavras reconhecíveis)
    # Abordagem: a marca são as N primeiras palavras até encontrar um token
    # que pareça parte do tipo de produto (contém maiúsculas internas ou é
    # uma palavra de tipo conhecida)
    TIPOS_CONHECIDOS = [
        'CRT TV', 'Color TV', 'LCD HDTV', 'Digital TV', 'Analog CRT TV',
        'MP3 Player', 'MP4 Player', 'Flash', 'Portable', 'Notebook',
        'Desktop', 'Laptop', 'Camera', 'Camcorder', 'Monitor',
        'Projector', 'Printer', 'Scanner', 'Mouse', 'Keyboard'
    ]
    tipo_encontrado = ''
    texto_restante = ' '.join(tokens)
    for tipo in sorted(TIPOS_CONHECIDOS, key=len, reverse=True):
        if tipo.lower() in texto_restante.lower():
            tipo_encontrado = tipo
            break
    resultado['Tipo_Produto'] = tipo_encontrado

    return resultado


def extrair_subcategoria(sub_raw):
    """
    Recebe: 'Subcategoria # 9 - Televisions'
    Retorna: (id_sub, nome_sub)
    """
    sub_raw = str(sub_raw).strip()
    m = re.search(r'#\s*(\d+)\s*-\s*(.+)', sub_raw)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return '', sub_raw

marca_atual = ''
registros = []
linhas_removidas = 0

for _, row in df.iterrows():
    prod_val = str(row['Produto_Raw']).strip()
    sub_val  = str(row['Subcategoria_Raw']).strip()

    if prod_val == '' or prod_val.lower().startswith('lista de'):
        linhas_removidas += 1
        continue

    if prod_val.lower().startswith('marca:'):
        partes = prod_val.split(':', 1)
        nome_marca = partes[1].strip() if len(partes) > 1 and partes[1].strip() else sub_val
        marca_atual = nome_marca
        linhas_removidas += 1
        continue

    campos = extrair_campos_produto(prod_val)
    id_sub, nome_sub = extrair_subcategoria(sub_val)

    registros.append({
        'ID_Produto'     : campos['ID_Produto'],
        'Nome_Produto'   : campos['Nome_Produto'],
        'Marca'          : marca_atual,
        'Tipo_Produto'   : campos['Tipo_Produto'],
        'Modelo'         : campos['Modelo'],
        'Cor'            : campos['Cor'],
        'Tamanho'        : campos['Tamanho'],
        'ID_Subcategoria': id_sub,
        'Subcategoria'   : nome_sub
    })

df_prod = pd.DataFrame(registros)

print(f'Linhas removidas (cabeçalhos/separadores): {linhas_removidas}')
print(f'Registros de produtos extraídos: {len(df_prod)}')
print(f'Marcas encontradas: {df_prod["Marca"].unique().tolist()}')
print(f'Subcategorias encontradas: {df_prod["Subcategoria"].unique().tolist()}')
df_prod.head(15)

# ─── Verificar campos obrigatórios vazios ─────────────────────────────────────
OBRIGATORIOS = ['ID_Produto', 'Nome_Produto', 'Marca', 'Tipo_Produto',
                'Modelo', 'ID_Subcategoria', 'Subcategoria']

print('=== Validação de campos obrigatórios ===')
for campo in OBRIGATORIOS:
    vazios = (df_prod[campo] == '').sum()
    status = '✅' if vazios == 0 else f'⚠️  {vazios} registro(s) vazio(s)'
    print(f'  {campo}: {status}')

print('\n=== Campos opcionais ===')
for campo in ['Cor', 'Tamanho']:
    preenchidos = (df_prod[campo] != '').sum()
    print(f'  {campo}: {preenchidos} de {len(df_prod)} preenchidos')

df_prod.head(10)

# ─── Formatar como texto e exportar XLSX ─────────────────────────────────────
for col in df_prod.columns:
    df_prod[col] = df_prod[col].astype(str).str.strip()

with pd.ExcelWriter(OUTPUT_PROD, engine='openpyxl') as writer:
    df_prod.to_excel(writer, index=False, sheet_name='Produto')
    ws = writer.sheets['Produto']
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.number_format = '@'

print(f'✅ Arquivo salvo em: {OUTPUT_PROD}')
print(f'Total de registros exportados: {len(df_prod)}')
df_prod.tail(5)

Separador detectado: ';'
Cabeçalho detectado na linha: 3
Shape inicial: (685, 2)
Linhas removidas (cabeçalhos/separadores): 29
Registros de produtos extraídos: 656
Marcas encontradas: ['Adventure Works', 'Contoso', 'Fabrikam', 'Litware', 'Northwind Traders', 'Proseware', 'Southridge Video', 'Wide World Importers']
Subcategorias encontradas: ['Televisions', 'Laptops', 'Desktops', 'Monitors', 'MP4&MP3', 'Home Theater System', 'Projectors & Screens', 'Bluetooth Headphones', 'Printers, Scanners & Fax', 'VCD & DVD', 'Car Video', 'Recording Pen']
=== Validação de campos obrigatórios ===
  ID_Produto: ✅
  Nome_Produto: ✅
  Marca: ✅
  Tipo_Produto: ⚠️  373 registro(s) vazio(s)
  Modelo: ⚠️  12 registro(s) vazio(s)
  ID_Subcategoria: ✅
  Subcategoria: ✅

=== Campos opcionais ===
  Cor: 656 de 656 preenchidos
  Tamanho: 42 de 656 preenchidos
✅ Arquivo salvo em: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Produto_tratado.xlsx
Total de registros exportados: 656


,ID_Produto,Nome_Produto,Marca,Tipo_Produto,Modelo,Cor,Tamanho,ID_Subcategoria,Subcategoria
651,636,WWI Projector 720p DLP56 Silver,Wide World Importers,Projector,DLP56,Silver,,19,Projectors & Screens
652,637,WWI Projector 480p LCD12 Silver,Wide World Importers,Projector,LCD12,Silver,,19,Projectors & Screens
653,638,WWI Projector 480p DLP12 Silver,Wide World Importers,Projector,DLP12,Silver,,19,Projectors & Screens
654,640,WWI Screen 113in M1610 Silver,Wide World Importers,,M1610,Silver,,19,Projectors & Screens
655,641,WWI Screen 106in M1609 Silver,Wide World Importers,,M1609,Silver,,19,Projectors & Screens


In [8]:
#VENDAS#

INPUT_VEND  = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Vendas.xlsx'
OUTPUT_VEND = os.path.join(OUTPUT_DIR, 'Vendas_tratada.xlsx')

# ─── Leitura bruta para detectar linha do cabeçalho ──────────────────────────
df_raw = pd.read_excel(INPUT_VEND, header=None, dtype=str)

header_idx = None
for i, row in df_raw.iterrows():
    vals = [str(v).strip().lower() for v in row.values]
    if 'data venda' in vals:
        header_idx = i
        break
if header_idx is None:
    header_idx = 5  # fallback: linha 6 (índice 5)
print(f'Cabeçalho detectado na linha: {header_idx + 1}')

# ─── Re-leitura sem dtype=str para preservar datas e números ─────────────────
df_vend = pd.read_excel(INPUT_VEND, header=header_idx)

# Remover colunas sem nome
df_vend = df_vend.loc[:, ~df_vend.columns.astype(str).str.contains('^Unnamed')]

# Remover linhas completamente vazias
df_vend = df_vend.dropna(how='all').reset_index(drop=True)

print('Colunas:', df_vend.columns.tolist())
print('Shape:', df_vend.shape)
df_vend.head(10)

# ─── Colunas monetárias e de data ─────────────────────────────────────────────
COLS_MOEDA = ['Custo Unitário', 'Preço Unitário', 'Valor Desconto']
COLS_DATA  = ['Data Venda', 'Data Envio']
FORMATO_MOEDA = 'R$ #,##0.00'
FORMATO_DATA  = 'DD/MM/YYYY'

# Converter monetários — já vêm como número do Excel, só garante float
for col in COLS_MOEDA:
    if col in df_vend.columns:
        df_vend[col] = pd.to_numeric(df_vend[col], errors='coerce')

# Garantir que datas sejam datetime
for col in COLS_DATA:
    if col in df_vend.columns:
        df_vend[col] = pd.to_datetime(df_vend[col], errors='coerce')

# ─── Exportar XLSX com formatações ───────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_VEND, engine='openpyxl', datetime_format='DD/MM/YYYY') as writer:
    df_vend.to_excel(writer, index=False, sheet_name='Vendas')

    ws = writer.sheets['Vendas']
    col_names = [cell.value for cell in ws[1]]

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            col_name = col_names[cell.column - 1]
            if col_name in COLS_MOEDA:
                cell.number_format = FORMATO_MOEDA
            elif col_name in COLS_DATA:
                cell.number_format = FORMATO_DATA
            else:
                cell.number_format = '@'

print(f'✅ Arquivo salvo em: {OUTPUT_VEND}')
print(f'Total de registros exportados: {len(df_vend)}')
df_vend.head(10)

Cabeçalho detectado na linha: 6
Colunas: ['Data Venda', 'Data Envio', 'ID Produto', 'ID Subcategoria', 'ID Cliente', 'ID Cidade', 'No. Venda', 'Custo Unitário', 'Preço Unitário', 'Quantidade', 'Valor Desconto']
Shape: (70055, 11)
✅ Arquivo salvo em: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Vendas_tratada.xlsx
Total de registros exportados: 70055


,Data Venda,Data Envio,ID Produto,ID Subcategoria,ID Cliente,ID Cidade,No. Venda,Custo Unitário,Preço Unitário,Quantidade,Valor Desconto
0,2017-01-01,2017-01-14,176,10,79,467,20070101811078,58.36,126.9,1,6.345
1,2017-01-01,2017-01-14,176,10,349,674,20070101411348,58.36,126.9,1,25.380
2,2017-01-01,2017-01-14,176,10,469,576,20070101511468,58.36,126.9,1,25.380
3,2017-01-01,2017-01-14,176,10,89,688,20070101311088,58.36,126.9,1,6.345
4,2017-01-01,2017-01-14,176,10,129,678,20070101511128,58.36,126.9,1,6.345
5,2017-01-01,2017-01-14,176,10,129,678,20070101811128,58.36,126.9,1,6.345
6,2017-01-01,2017-01-14,176,10,159,690,20070101311158,58.36,126.9,1,6.345
7,2017-01-01,2017-01-14,176,10,389,576,20070101511388,58.36,126.9,1,25.380
8,2017-01-01,2017-01-14,176,10,389,576,20070101411388,58.36,126.9,1,25.380
9,2017-01-01,2017-01-14,176,10,389,576,20070101811388,58.36,126.9,1,25.380


In [9]:
import json as json_lib

INPUT_SUB  = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Subcategoria.json'
OUTPUT_SUB = os.path.join(OUTPUT_DIR, 'Subcategoria_tratada.xlsx')

# ─── Leitura do JSON ──────────────────────────────────────────────────────────
with open(INPUT_SUB, 'r', encoding='utf-8-sig') as f:
    dados = json_lib.load(f)

df_sub = pd.DataFrame(dados)

print('Colunas:', df_sub.columns.tolist())
print('Shape:', df_sub.shape)
df_sub.head(10)

# ─── Renomear colunas para padrão consistente ─────────────────────────────────
df_sub = df_sub.rename(columns={
    'idSubcategoria': 'ID_Subcategoria',
    'Subcategoria'  : 'Subcategoria',
    'Categoria'     : 'Categoria'
})

# ─── Validar campos obrigatórios ──────────────────────────────────────────────
OBRIGATORIOS = ['ID_Subcategoria', 'Subcategoria', 'Categoria']
print('=== Validação de campos obrigatórios ===')
for campo in OBRIGATORIOS:
    nulos = df_sub[campo].isna().sum()
    vazios = (df_sub[campo].astype(str).str.strip() == '').sum()
    status = '✅' if (nulos + vazios) == 0 else f'⚠️  {nulos + vazios} registro(s) vazio(s)'
    print(f'  {campo}: {status}')

# ─── Formatar todos como texto ────────────────────────────────────────────────
for col in df_sub.columns:
    df_sub[col] = df_sub[col].astype(str).str.strip()

# ─── Exportar como XLSX ──────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_SUB, engine='openpyxl') as writer:
    df_sub.to_excel(writer, index=False, sheet_name='Subcategoria')
    ws = writer.sheets['Subcategoria']
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.number_format = '@'

print(f'\n✅ Arquivo salvo em: {OUTPUT_SUB}')
print(f'Total de registros exportados: {len(df_sub)}')
df_sub

Colunas: ['idSubcategoria', 'Subcategoria', 'Categoria']
Shape: (44, 3)
=== Validação de campos obrigatórios ===
  ID_Subcategoria: ✅
  Subcategoria: ✅
  Categoria: ✅

✅ Arquivo salvo em: C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Subcategoria_tratada.xlsx
Total de registros exportados: 44


,ID_Subcategoria,Subcategoria,Categoria
0,1,MP4&MP3,Audio
1,2,Recorder,Audio
2,3,Radio,Audio
3,4,Recording Pen,Audio
4,5,Headphones,Audio
5,6,Bluetooth Headphones,Audio
6,7,Speakers,Audio
7,8,Audio Accessories,Audio
8,9,Televisions,TV and Video
9,10,VCD & DVD,TV and Video


In [18]:
# ─── Exportar todos os XLSX tratados como CSV (sep=';') ───────────────────────

OUTPUT_CSV_DIR = r'C:\Users\AdminUser\Downloads\desafios_vendas_x_metas\Bases_tratadas\Arquivos.CSV'
os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)

arquivos_xlsx = [
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith('.xlsx')
]

print(f'Arquivos XLSX encontrados: {len(arquivos_xlsx)}')
print('-' * 50)

for nome_xlsx in sorted(arquivos_xlsx):
    caminho_xlsx = os.path.join(OUTPUT_DIR, nome_xlsx)
    nome_csv     = nome_xlsx.replace('.xlsx', '.csv')
    caminho_csv  = os.path.join(OUTPUT_CSV_DIR, nome_csv)

    # Lê o XLSX tratado
    df_temp = pd.read_excel(caminho_xlsx, dtype=str)

    # Salva como CSV com separador ';' e encoding UTF-8 com BOM (acentos ok)
    df_temp.to_csv(
        caminho_csv,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

    print(f'✅ {nome_csv} — {len(df_temp)} registros exportados')

print('-' * 50)
print(f'\n✅ Todos os CSVs salvos em: {OUTPUT_CSV_DIR}')

Arquivos XLSX encontrados: 7
--------------------------------------------------
✅ Cliente.csv — 18869 registros exportados
✅ Localizacao_tratada.csv — 671 registros exportados
✅ Metas_tratada.csv — 27 registros exportados
✅ Produto_tratado.csv — 656 registros exportados
✅ Subcategoria_tratada.csv — 44 registros exportados
✅ Vendas_tratada.csv — 70055 registros exportados


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\AdminUser\\Downloads\\desafios_vendas_x_metas\\Bases_tratadas\\~$Subcategoria_tratada.xlsx'